In [ ]:
import cv2
import os
import glob
import numpy as np
from collections import defaultdict

query_image_path = "../data/test_images/my_messy_500_peso_photo.jpg" 
database_dir = "../data/database/orb_database/"
RATIO_THRESH = 0.75
MIN_MATCHES = 10

def recognize_banknote(query_path, db_dir):
    print(f"Analizando: {os.path.basename(query_path)}")
    
    query_img = cv2.imread(query_path)
    if query_img is None:
        print("Error: No se pudo cargar la imagen de consulta.")
        return
        
    gray_query = cv2.cvtColor(query_img, cv2.COLOR_BGR2GRAY)
    
    orb = cv2.ORB_create(nfeatures=2500)
    kp_query, des_query = orb.detectAndCompute(gray_query, None)
    
    if des_query is None:
        print("No se encontraron características en la imagen.")
        return

    # BFMatcher con NORM_HAMMING para descriptores binarios (ORB)
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False) 
    category_votes = defaultdict(int)
    db_files = glob.glob(os.path.join(db_dir, "*.npy"))
    
    if not db_files:
        print("Error: Base de datos .npy no encontrada.")
        return
    
    for db_file in db_files:
        des_db = np.load(db_file)
        if des_db is None or len(des_db) == 0:
            continue
            
        filename = os.path.basename(db_file)
        category = filename.split("_comp_")[0].replace("norm_clean_", "")
        
        matches = matcher.knnMatch(des_query, des_db, k=2)
        
        # Conteo de coincidencias mediante Ratio Test de Lowe
        good_matches = sum(1 for m_n in matches 
                           if len(m_n) == 2 and m_n[0].distance < RATIO_THRESH * m_n[1].distance)
                    
        if good_matches >= MIN_MATCHES:
            category_votes[category] += good_matches
            print(f"  {good_matches} coincidencias con: {category}")

    # Resultados finales
    if not category_votes:
        print("Resultado: Billete no reconocido.")
    else:
        results = sorted(category_votes.items(), key=lambda x: x[1], reverse=True)
        winner, score = results[0]
        
        print("-" * 40)
        print(f"BILLETE DETECTADO: {winner}")
        print(f"Total de coincidencias: {score}")
        print("-" * 40)
        
        if len(results) > 1:
            print(f"Segundo candidato: {results[1][0]} ({results[1][1]} matches)")

recognize_banknote(query_image_path, database_dir)

--- Analizando Imagen de Consulta: my_messy_500_peso_photo.jpg ---
  7 coincidencias con el componente 1000PesosBack.
  31 coincidencias con el componente 1000PesosFront.
  9 coincidencias con el componente 1000PesosFront.
  56 coincidencias con el componente 100PesosBack.
  15 coincidencias con el componente 100PesosBack.
  59 coincidencias con el componente 100PesosFront.
  7 coincidencias con el componente 100PesosFront.
  21 coincidencias con el componente 200PesosBack.
  10 coincidencias con el componente 200PesosBack.
  16 coincidencias con el componente 200PesosFront.
  26 coincidencias con el componente 20PesosBack.
  6 coincidencias con el componente 20PesosBack.
  28 coincidencias con el componente 20PesosFront.
  41 coincidencias con el componente 20PesosPolimeroBack.
  8 coincidencias con el componente 20PesosPolimeroBack.
  8 coincidencias con el componente 20PesosPolimeroFront.
  11 coincidencias con el componente 20PesosPolimeroFront.
  17 coincidencias con el componente